# Day 2 — Feature Engineering & Target Construction
YouTube Real-Time Engagement Prediction Project

Goal for today:
1. Load Day 1's cleaned data (one row per snapshot)
2. For each video, extract EARLY-WINDOW features (from snapshots <= 6h old)
3. For each video, construct the TARGET (engagement_rate closest to 24h,
   accepted within an 18-30h window)
4. Merge into ONE row per video — this becomes the actual ML training table
5. Save it for Day 3 (modeling)

NOTE: Title-based features are intentionally EXCLUDED. With only ~160
unique videos, text-derived features carry a high risk of overfitting to
sample-specific patterns rather than learning generalizable relationships.
Numeric/categorical features are preferred for their statistical reliability
at this sample size.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

PROCESSED_PATH = "C:/Users/Lenovo/youtube_enagagement_project/data/processed/03cleaned_data_v2_upgrade.csv"
TRAINING_TABLE_PATH = "C:/Users/Lenovo/youtube_enagagement_project//data/processed/03training_table_v2_upgrade.csv"

EARLY_CUTOFF = 6          # hours — features built from snapshots at or before this age
TARGET_LOW, TARGET_HIGH = 18, 30   # hours — window accepted as "~24h" target

df = pd.read_csv(PROCESSED_PATH)
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df["snapshot_time"] = pd.to_datetime(df["snapshot_time"], utc=True)

print("Shape:", df.shape)
print("Unique videos:", df["video_id"].nunique())

Shape: (96825, 16)
Unique videos: 537


## 1. Time-based features (from published_at — known at upload time)
These are static per video, so we can compute them directly on the full df
then just take one value per video later.

In [2]:
df["upload_hour"] = df["published_at"].dt.hour
df["upload_dayofweek"] = df["published_at"].dt.dayofweek       # 0=Monday
df["is_weekend"] = df["upload_dayofweek"].isin([5, 6]).astype(int)

## 2. Build EARLY-WINDOW features (one row per video)
For each video, take the snapshot CLOSEST TO (but not exceeding) the early
cutoff — this gives us the most information available while still being
"early". This snapshot's views/likes/comments/rates become our features.

In [3]:
early_df = df[df["video_age_hours"] <= EARLY_CUTOFF].copy()

# For each video, pick the row with the LARGEST video_age_hours within the
# early window (i.e. the most recent snapshot still inside the early period)
early_idx = early_df.groupby("video_id")["video_age_hours"].idxmax()
early_features = early_df.loc[early_idx].copy()

# Rename to make clear these are "early" values
early_features = early_features.rename(columns={
    "views": "early_views",
    "likes": "early_likes",
    "comments": "early_comments",
    "views_per_hour": "early_views_per_hour",
    "like_rate": "early_like_rate",
    "comment_rate": "early_comment_rate",
    "engagement_rate": "early_engagement_rate",
    "video_age_hours": "early_snapshot_age_hours",
})

# Count how many early snapshots each video had (a simple data-density feature)
early_snapshot_counts = early_df.groupby("video_id").size().rename("early_snapshot_count")

early_features = early_features.merge(early_snapshot_counts, on="video_id")

# Keep only the columns we need from the early snapshot
early_cols = [
    "video_id", "category_id", "duration_seconds",
    "upload_hour", "upload_dayofweek", "is_weekend",
    "early_snapshot_age_hours", "early_snapshot_count",
    "early_views", "early_likes", "early_comments",
    "early_views_per_hour", "early_like_rate", "early_comment_rate",
    "early_engagement_rate",
]
early_features = early_features[early_cols]

print("Early features shape:", early_features.shape)
early_features.head()

Early features shape: (537, 15)


,video_id,category_id,duration_seconds,upload_hour,upload_dayofweek,is_weekend,early_snapshot_age_hours,early_snapshot_count,early_views,early_likes,early_comments,early_views_per_hour,early_like_rate,early_comment_rate,early_engagement_rate
0,-0ECJhNocZQ,24.0,557.0,9,6,1,5.941468,27,484,75,10,81.461345,15.495868,2.066116,17.561983
1,-2Qau5p06MI,27.0,599.0,6,3,0,5.943943,30,102,8,1,17.160326,7.843137,0.980392,8.823529
2,-8Irr9Ah2i4,15.0,179.0,21,1,0,5.879336,32,223455,16779,1111,38006.842231,7.508894,0.497192,8.006086
3,-ATKM6TIgB4,15.0,68.0,22,4,0,5.942039,33,1569,14,0,264.050792,0.892288,0.000000,0.892288
4,-Bfjyx-VaLs,15.0,120.0,20,1,0,5.862097,26,25568,5026,464,4361.578769,19.657384,1.814768,21.472153


## 3. Build the TARGET — engagement_rate closest to 24h
For each video, find the snapshot within the 18-30h window whose
video_age_hours is closest to 24. Use its engagement_rate as the target.

In [4]:
target_window_df = df[(df["video_age_hours"] >= TARGET_LOW) &
                       (df["video_age_hours"] <= TARGET_HIGH)].copy()

target_window_df["distance_from_24h"] = (target_window_df["video_age_hours"] - 24).abs()

target_idx = target_window_df.groupby("video_id")["distance_from_24h"].idxmin()
target_rows = target_window_df.loc[target_idx].copy()

target_rows = target_rows.rename(columns={
    "engagement_rate": "target_engagement_rate",
    "video_age_hours": "target_snapshot_age_hours",
})

target_cols = ["video_id", "target_snapshot_age_hours", "target_engagement_rate"]
target_rows = target_rows[target_cols]

print("Target rows shape:", target_rows.shape)
target_rows.head()

Target rows shape: (537, 3)


,video_id,target_snapshot_age_hours,target_engagement_rate
134,-0ECJhNocZQ,23.941531,3.951207
281,-2Qau5p06MI,23.943957,4.979253
461,-8Irr9Ah2i4,24.045936,4.969217
649,-ATKM6TIgB4,23.942068,0.660851
832,-Bfjyx-VaLs,24.028786,15.218965


## 3b. Exclude videos with implausible target values
A small number of videos have a target-window engagement rate that's
still implausibly high (e.g. >50%) even after the cummax fix in Day 1.
The most likely cause is YouTube's view-counting delay: a view requires
a minimum watch time to count, while a like/comment registers instantly
— so a small-audience video can briefly show more likes+comments than
counted views. Rather than distorting these values with an artificial
cap (which would corrupt the regression target with a false ceiling),
we exclude the affected videos entirely and report exactly how many.

In [5]:
IMPLAUSIBLE_THRESHOLD = 50.0  # % — generous ceiling; genuine viral engagement rarely exceeds this

implausible_videos = target_rows[target_rows["target_engagement_rate"] > IMPLAUSIBLE_THRESHOLD]["video_id"].tolist()
print(f"Excluding {len(implausible_videos)} videos with target_engagement_rate > {IMPLAUSIBLE_THRESHOLD}%:")
print(implausible_videos)

target_rows = target_rows[~target_rows["video_id"].isin(implausible_videos)]
print("\nTarget rows shape after exclusion:", target_rows.shape)

Excluding 13 videos with target_engagement_rate > 50.0%:
['4pD6d1kW_4o', '8Z14iueS4eM', 'DkokDagKe-U', 'T46isS4shGE', 'Yut2QtmUJDI', 'b1FRQgDEcG4', 'cDR74Ykyb-I', 'hlnq6Aplmno', 'i32mj6s9Nd8', 'ji09Gjag9LI', 'm5jlM1tGfLM', 'qFnUALceXg4', 'zaB1HO--jN4']

Target rows shape after exclusion: (524, 3)


## 4. Merge features + target into ONE training table (one row per video)

In [6]:
training_table = early_features.merge(target_rows, on="video_id", how="inner")

print("Final training table shape:", training_table.shape)
print("Unique videos:", training_table["video_id"].nunique())
training_table.head()

Final training table shape: (524, 17)
Unique videos: 524


,video_id,category_id,duration_seconds,upload_hour,upload_dayofweek,is_weekend,early_snapshot_age_hours,early_snapshot_count,early_views,early_likes,early_comments,early_views_per_hour,early_like_rate,early_comment_rate,early_engagement_rate,target_snapshot_age_hours,target_engagement_rate
0,-0ECJhNocZQ,24.0,557.0,9,6,1,5.941468,27,484,75,10,81.461345,15.495868,2.066116,17.561983,23.941531,3.951207
1,-2Qau5p06MI,27.0,599.0,6,3,0,5.943943,30,102,8,1,17.160326,7.843137,0.980392,8.823529,23.943957,4.979253
2,-8Irr9Ah2i4,15.0,179.0,21,1,0,5.879336,32,223455,16779,1111,38006.842231,7.508894,0.497192,8.006086,24.045936,4.969217
3,-ATKM6TIgB4,15.0,68.0,22,4,0,5.942039,33,1569,14,0,264.050792,0.892288,0.000000,0.892288,23.942068,0.660851
4,-Bfjyx-VaLs,15.0,120.0,20,1,0,5.862097,26,25568,5026,464,4361.578769,19.657384,1.814768,21.472153,24.028786,15.218965


## 5. Sanity check — confirm no leakage columns remain
We should NOT have: views, likes, comments, like_rate, comment_rate,
engagement_rate (the raw/current-time versions) as features — only the
"early_" prefixed versions and the target are allowed.

In [7]:
leakage_risk_cols = ["views", "likes", "comments", "like_rate", "comment_rate", "engagement_rate"]
present = [c for c in leakage_risk_cols if c in training_table.columns]
print("Leakage-risk columns present (should be empty):", present)

Leakage-risk columns present (should be empty): []


## 6. Quick check of target distribution

In [8]:
print(training_table["target_engagement_rate"].describe())

count    524.000000
mean       4.952027
std        6.203134
min        0.000000
25%        1.099318
50%        2.951167
75%        6.234188
max       50.000000
Name: target_engagement_rate, dtype: float64


## 7. Save the final training table for Day 3 (modeling)

In [9]:
import os
os.makedirs(os.path.dirname(TRAINING_TABLE_PATH), exist_ok=True)
training_table.to_csv(TRAINING_TABLE_PATH, index=False)
print(f"Saved training table to {TRAINING_TABLE_PATH}")
print(training_table.shape)

Saved training table to C:/Users/Lenovo/youtube_enagagement_project//data/processed/03training_table_v2_upgrade.csv
(524, 17)
